In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA = Path("../data/processed/criteo_uplift_v2.1.parquet")
OUT = Path("../reports"); OUT.mkdir(exist_ok=True)

df = pd.read_parquet(DATA)
FEATURES = [f"f{i}" for i in range(12)]
OUTCOMES = ["visit", "conversion"]
Z = 1.959964  # 97,5%-Quantil der Standardnormalverteilung (für 95%-KI)

In [3]:
def diff_in_means(y, w):
    """Differenz der Mittelwerte mit Neyman-Standardfehler
    sowie relativer Lift mit Delta-Methode."""
    y1, y0 = y[w == 1], y[w == 0]
    m1, m0 = y1.mean(), y0.mean()
    v1 = y1.var(ddof=1) / len(y1)   # Varianz des Mittelwerts, Treatment
    v0 = y0.var(ddof=1) / len(y0)   # Varianz des Mittelwerts, Kontrolle
    tau = m1 - m0
    se = np.sqrt(v1 + v0)
    lift = m1 / m0 - 1
    se_lift = np.sqrt(v1 / m0**2 + m1**2 * v0 / m0**4)  # Delta-Methode
    return dict(base=m0, ate=tau, se=se, lift=lift, se_lift=se_lift)

w = df["treatment"].to_numpy()
dim = {y: diff_in_means(df[y].to_numpy(dtype=np.float64), w) for y in OUTCOMES}
pd.DataFrame(dim).T

,base,ate,se,lift,se_lift
visit,0.038201,0.010342,0.000146,0.270737,0.004696
conversion,0.001938,0.001152,0.000034,0.594488,0.026336


In [4]:
def lin_estimator(df, y_name, features, chunk=2_000_000):
    """Regressionsadjustierter ATE nach Lin (2013):
    OLS von y auf [1, W, Xc, W*Xc] mit zentrierten Kovariaten, HC1-Standardfehler.
    Blockweise Berechnung, damit der Speicherbedarf gering bleibt."""
    mu = df[features].mean().to_numpy()
    n, k = len(df), 2 + 2 * len(features)

    def design(block):
        xc = block[features].to_numpy(dtype=np.float64) - mu
        wb = block["treatment"].to_numpy(dtype=np.float64)[:, None]
        return np.hstack([np.ones((len(block), 1)), wb, xc, wb * xc])

    # Erster Durchlauf: X'X und X'y akkumulieren
    xtx, xty = np.zeros((k, k)), np.zeros(k)
    for s in range(0, n, chunk):
        b = df.iloc[s:s + chunk]
        X = design(b)
        xtx += X.T @ X
        xty += X.T @ b[y_name].to_numpy(dtype=np.float64)
    beta = np.linalg.solve(xtx, xty)

    # Zweiter Durchlauf: "Meat" der Sandwich-Varianz
    meat = np.zeros((k, k))
    for s in range(0, n, chunk):
        b = df.iloc[s:s + chunk]
        X = design(b)
        e = b[y_name].to_numpy(dtype=np.float64) - X @ beta
        Xe = X * e[:, None]
        meat += Xe.T @ Xe

    bread = np.linalg.inv(xtx)
    cov = bread @ meat @ bread * n / (n - k)   # HC1-Korrektur
    return beta[1], np.sqrt(cov[1, 1])         # Koeffizient von W = ATE

In [5]:
rows = []
for y in OUTCOMES:
    tau_adj, se_adj = lin_estimator(df, y, FEATURES)
    d = dim[y]
    rows.append({
        "outcome": y,
        "base_rate_control": d["base"],
        "ate_dim": d["ate"], "se_dim": d["se"],
        "ate_lin": tau_adj, "se_lin": se_adj,
        "lift": d["lift"],
        "lift_ci_low": d["lift"] - Z * d["se_lift"],
        "lift_ci_high": d["lift"] + Z * d["se_lift"],
    })
res = pd.DataFrame(rows).set_index("outcome")

res["se_ratio"] = res["se_lin"] / res["se_dim"]
# Äquivalente Einsparung an Stichprobengröße bei gleicher Präzision
res["sample_saving"] = 1 - res["se_ratio"] ** 2
res.round(6)

,base_rate_control,ate_dim,se_dim,ate_lin,se_lin,lift,lift_ci_low,lift_ci_high,se_ratio,sample_saving
outcome,,,,,,,,,,
visit,0.038201,0.010342,0.000146,0.007734,0.000131,0.270737,0.261533,0.279941,0.897065,0.195274
conversion,0.001938,0.001152,0.000034,0.001002,0.000034,0.594488,0.542871,0.646105,0.988485,0.022898


In [6]:
def wald_iv(y, d, z):
    """IV-Schätzer (Wald) mit einem Instrument, ohne Kovariaten.
    Robuster Standardfehler (HC0) für den exakt identifizierten Fall."""
    yt, dt, zt = y - y.mean(), d - d.mean(), z - z.mean()
    beta = (zt @ yt) / (zt @ dt)
    u = yt - beta * dt
    se = np.sqrt(np.sum(zt**2 * u**2)) / abs(zt @ dt)
    return beta, se

z = df["treatment"].to_numpy(dtype=np.float64)
d = df["exposure"].to_numpy(dtype=np.float64)

# Erste Stufe: Anteil der tatsächlich exponierten Nutzer (Compliance-Rate)
compliance = d[z == 1].mean() - d[z == 0].mean()
print(f"Compliance-Rate: {compliance:.4f}")

iv_rows = []
for y_name in OUTCOMES:
    y = df[y_name].to_numpy(dtype=np.float64)
    att, se_att = wald_iv(y, d, z)
    itt = res.loc[y_name, "ate_dim"]
    iv_rows.append({
        "outcome": y_name,
        "itt": itt,
        "att_exposed": att, "se_att": se_att,
        "att_ci_low": att - Z * se_att, "att_ci_high": att + Z * se_att,
        "check_itt_div_compliance": itt / compliance,  # Muss ≈ att sein (Bloom)
    })
iv = pd.DataFrame(iv_rows).set_index("outcome")
iv.round(6)

Compliance-Rate: 0.0360


,itt,att_exposed,se_att,att_ci_low,att_ci_high,check_itt_div_compliance
outcome,,,,,,
visit,0.010342,0.286996,0.004022,0.279112,0.294880,0.286996
conversion,0.001152,0.031964,0.000951,0.030100,0.033828,0.031964


In [7]:
res.to_csv(OUT / "ate_results.csv")
iv.to_csv(OUT / "iv_results.csv")

## Findings
- ATE visit (Differenz der Mittelwerte): ___ (95%-KI ___ bis ___), relativer Lift ___
- ATE conversion: ___ (95%-KI ___ bis ___), relativer Lift ___
- Regressionsadjustierung: Standardfehler um ___% reduziert,
  entspricht ___% weniger Stichprobe bei gleicher Präzision
- Compliance-Rate (Anteil exponierter Nutzer in der Treatment-Gruppe): ___
- Effekt auf exponierte Nutzer (ATT, visit): ___ (95%-KI ___ bis ___)
- Hinweis: Absolute Effektgrößen sind wegen der datenschutzbedingten
  Unterstichprobe nicht als reale Werbewirkung zu interpretieren